# 01. Python + NumPy + ООП: каталог экзопланет — домашняя работа

В этом ноутбуке мы построим маленький инструмент для анализа астрономических данных,
а затем применим его к **реальному каталогу подтверждённых экзопланет NASA Exoplanet Archive**.

**Основная идея:** сначала мы делаем часть работы «вручную» с Python и NumPy,
затем упаковываем её в собственный класс `ExoplanetDataset`. В следующем ноутбуке
похожие операции будут выполняться уже с помощью Polars.

### Что повторяем
- списки, словари, условия, циклы, функции;
- `try/except`, `enumerate`, `zip`, comprehensions;
- массивы NumPy, индексацию и boolean masks;
- `np.nan`, статистики и стандартизацию;
- основы ООП: `class`, `self`, `__init__`, `__len__`, `__repr__`, `__getitem__`,
  `@property`, `@classmethod`;
- чтение CSV;
- физический смысл некоторых параметров экзопланет.

> **Правило:** до раздела с NASA не используйте `pandas`, `polars` и `scikit-learn`.
> Наша цель — понять, что именно библиотеки будут делать за нас позже.

## 0. Импорты

В основной части нам нужны только стандартная библиотека Python и NumPy.

### Материалы к разделу

Перед выполнением достаточно просмотреть первые примеры; читать документацию целиком не требуется.

- [NumPy: absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) — что такое `ndarray`, создание массивов, `shape`, индексация и основные операции.
- [Python Tutorial](https://docs.python.org/3/tutorial/) — официальный учебник Python; полезен, если нужно вспомнить базовый синтаксис.
- [NumPy documentation](https://numpy.org/doc/stable/) — главная страница документации, откуда удобно искать функцию по имени.

In [ ]:
import csv
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlretrieve

import numpy as np

np.set_printoptions(precision=3, suppress=True)

## 1. Разминка: маленький каталог планет

Пока без NASA. Представим, что мы уже нашли несколько планет и записали четыре признака:
название, радиус в радиусах Земли, орбитальный период в сутках и равновесную температуру.

### Материалы к разделу

- [NumPy: indexing and slicing](https://numpy.org/doc/stable/user/absolute_beginners.html#indexing-and-slicing) — обычная и булева индексация.
- [NumPy: indexing on ndarrays](https://numpy.org/doc/stable/user/basics.indexing.html) — подробная документация по индексам и boolean masks.
- [NumPy quickstart: indexing with boolean arrays](https://numpy.org/doc/stable/user/quickstart.html#indexing-with-boolean-arrays) — короткие примеры фильтрации по условиям.

Особенно обратите внимание на выражения вида `a[a < 5]` и на объединение условий через `&` и `|`.

> **Подсказка к заданию 1.** Для поиска объекта по экстремальному значению сначала получите индекс через `np.argmax` или `np.argmin`, а затем используйте этот индекс для массива `names`. Для двух условий над NumPy-массивом используйте `(condition_1) & (condition_2)`.

In [ ]:
names = np.array(["Astra-1 b", "Astra-2 b", "Astra-3 b", "Astra-4 b", "Astra-5 b"])
radius = np.array([1.10, 2.40, 11.20, 0.80, 1.55])
period = np.array([365.0, 12.3, 4.1, 89.2, 42.0])
temperature = np.array([280.0, 740.0, 1300.0, 340.0, 310.0])

print(names)

### Задание 1

1. Найдите название самой большой планеты.
2. Найдите название самой горячей планеты.
3. Получите массив названий планет с радиусом меньше `2 R⊕`.
4. Получите массив названий планет с температурой от 200 до 350 K включительно.
5. Создайте словарь `period_by_name`, где ключ — название, значение — орбитальный период.

Для пунктов 1–4 постарайтесь не использовать Python-цикл.

In [ ]:
largest_name = ...
hottest_name = ...
small_names = ...
temperate_names = ...
period_by_name = ...

print("Самая большая:", largest_name)
print("Самая горячая:", hottest_name)
print("Маленькие:", small_names)
print("Умеренная температура:", temperate_names)
print("Периоды:", period_by_name)

### Самопроверка

In [ ]:
assert largest_name == "Astra-3 b"
assert hottest_name == "Astra-3 b"
assert set(small_names) == {"Astra-1 b", "Astra-4 b", "Astra-5 b"}
assert set(temperate_names) == {"Astra-1 b", "Astra-4 b", "Astra-5 b"}
assert period_by_name["Astra-2 b"] == 12.3
print("OK: Задание 1 пройдено")

## 2. Генерируем собственный каталог

Создадим 1000 искусственных планет. Мы не пытаемся получить идеальную модель реальной
популяции экзопланет: этот набор нужен для тренировки NumPy и ООП.

Большая полуось `a` задаётся в а.е., масса звезды — в массах Солнца.
Период вычислим приближённо из третьего закона Кеплера:

$
P_{\mathrm{year}} = \sqrt{\frac{a^3}{M_\star}}.
$

Затем добавим небольшой шум измерений.

### Материалы к разделу

- [NumPy random sampling](https://numpy.org/doc/stable/reference/random/index.html) — `default_rng`, seed и генерация случайных выборок.
- [NumPy Random Generator](https://numpy.org/doc/stable/reference/random/generator.html) — методы `normal`, `uniform`, `lognormal` и другие распределения.
- [numpy.argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) — как получить индексы, которые сортируют массив.
- [NumPy beginner guide](https://numpy.org/doc/stable/user/absolute_beginners.html) — операции над массивами без Python-циклов.

Полезная идея: boolean-массив состоит из `True` и `False`; его среднее равно доле элементов, для которых условие истинно.

> **Подсказка к заданию 2.** `np.argsort(x)` возвращает не отсортированный массив, а индексы в нужном порядке. Для подсчёта количества `True` можно использовать `np.sum(mask)`, а для доли — воспользоваться тем, что в вычислениях `True` ведёт себя как `1`, а `False` как `0`.

In [ ]:
rng = np.random.default_rng(42)
n = 1000

synthetic_names = np.array([f"SIM-{i:04d} b" for i in range(n)], dtype=object)

st_mass = np.clip(rng.normal(loc=1.0, scale=0.25, size=n), 0.15, None)
st_teff = np.clip(rng.normal(loc=5600, scale=900, size=n), 2500, 10000)

pl_orbsmax = 10 ** rng.uniform(-2.0, 0.7, size=n)
period_years_ideal = np.sqrt(pl_orbsmax**3 / st_mass)
pl_orbper = period_years_ideal * 365.25 * rng.normal(1.0, 0.02, size=n)

pl_rade = np.clip(rng.lognormal(mean=np.log(2.0), sigma=0.75, size=n), 0.2, 25)
pl_eqt = np.clip(278 * np.sqrt(np.sqrt(st_mass) / pl_orbsmax), 50, 3500)

synthetic_data = {
    "pl_name": synthetic_names,
    "pl_rade": pl_rade,
    "pl_orbper": pl_orbper,
    "pl_orbsmax": pl_orbsmax,
    "pl_eqt": pl_eqt,
    "st_mass": st_mass,
    "st_teff": st_teff,
}

{key: value.shape for key, value in synthetic_data.items()}

### Задание 2. NumPy без циклов

Для `pl_rade`:

1. найдите среднее, медиану и стандартное отклонение;
2. найдите индексы пяти планет с наименьшим радиусом;
3. посчитайте число планет с `R < 2 R⊕`;
4. посчитайте долю таких планет;
5. найдите планеты, у которых одновременно `R < 2 R⊕` и `200 <= T_eq <= 350 K`.

Дополнительное задание Для последнего пункта выведите не только количество, но и их названия.

In [ ]:
mean_radius = ...
median_radius = ...
std_radius = ...

five_smallest_idx = ...
n_small = ...
share_small = ...

candidate_mask = ...
candidate_names = ...

print("mean =", mean_radius)
print("median =", median_radius)
print("std =", std_radius)
print("5 smallest indices =", five_smallest_idx)
print("small planets:", n_small, f"({share_small:.1%})")
print("candidates:", candidate_names[:20])

## 3. «Грязные» данные

Настоящие научные каталоги содержат пропуски и неоднородные измерения.
Добавим несколько проблем в искусственные данные и научимся их находить.

### Материалы к разделу

- [NumPy statistics](https://numpy.org/doc/stable/reference/routines.statistics.html) — `mean`, `median`, `std`, `nanmean`, `nanmedian`, `nanstd`.
- [numpy.nanstd](https://numpy.org/doc/stable/reference/generated/numpy.nanstd.html) — пример статистики с игнорированием `NaN`.
- [Stack Overflow: ignore NaN in NumPy functions](https://stackoverflow.com/questions/66289714/ignore-nan-in-numpy-functions) — почему обычные `mean` и `std` возвращают `NaN` и какие функции использовать вместо них.
- [Python: Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html) — пригодится позже при обработке неправильных типов и значений.

Не путайте `np.nan` с нулём: `NaN` означает отсутствие или неопределённость числового значения.

> **Подсказка к заданию 3.** Сначала преобразуйте вход к числовому `np.ndarray`. Для стандартизации вычислите `np.nanmean(x)` и `np.nanstd(x)`, а затем примените формулу сразу ко всему массиву. Сравнивать каждый элемент циклом не требуется.

In [ ]:
dirty_data = {key: value.copy() for key, value in synthetic_data.items()}

dirty_data["pl_rade"][[17, 213, 581]] = np.nan
dirty_data["pl_eqt"][[42, 512]] = np.nan

dirty_data["pl_rade"][341] = 500.0
dirty_data["pl_eqt"][56] = -100.0

### Задание 3. Вспомогательные функции

Реализуйте:

- `count_missing(x)` — количество `NaN`;
- `zscore(x)` — стандартизация с игнорированием `NaN`;
- `outlier_mask(x, threshold=3)` — маска объектов, для которых `|z| > threshold`.

\[
z = \frac{x-\mu}{\sigma}.
\]

In [ ]:
def count_missing(x):
    # TODO
    pass

def zscore(x):
    # TODO
    pass

def outlier_mask(x, threshold=3):
    # TODO
    pass

In [ ]:
assert count_missing(dirty_data["pl_rade"]) == 3

z = zscore(np.array([1.0, 2.0, 3.0, np.nan]))
assert np.isclose(np.nanmean(z), 0.0)
assert np.isclose(np.nanstd(z), 1.0)

assert outlier_mask(dirty_data["pl_rade"]).dtype == bool
assert outlier_mask(dirty_data["pl_rade"]).shape == (n,)

print("OK: Вспомогательные функции работают")

## 4. Пишем собственный `ExoplanetDataset`

Внутри `data` хранится словарь `{"column": np.array(...)}`. Все столбцы должны иметь одинаковую длину.

Реализуйте `__init__`, `__len__`, `__repr__`, `columns`, `shape`, `__getitem__`,
`get_row`, `filter`, `add_column`, статистические методы, `unique`, `value_counts`,
`standardize`, а затем Дополнительное задание `from_csv`.

**Важно:** `filter` и `add_column` не должны менять исходный объект.

### Материалы к разделу

- [Python Tutorial: Classes](https://docs.python.org/3/tutorial/classes.html) — официально про классы, экземпляры, методы и атрибуты.
- [Real Python: Object-Oriented Programming in Python](https://realpython.com/python3-object-oriented-programming/) — более подробное объяснение ООП на примерах.
- [Real Python: Classes in Python, video](https://realpython.com/videos/classes-python/) — короткое видео о классах и экземплярах.
- [Python built-in functions: `classmethod`](https://docs.python.org/3/library/functions.html#classmethod) — что именно получает `cls`.
- [Stack Overflow: class methods as alternative constructors](https://stackoverflow.com/questions/73569542/how-class-methods-can-be-alternative-constructors-as-they-just-return-us-objects) — полезно перед реализацией `from_csv`.

При работе над классом не пытайтесь реализовать всё сразу: сначала добейтесь прохождения тестов для `__init__`, `__len__`, `columns` и `shape`, затем переходите к остальным методам.

> **Подсказка к классу.** В `__len__` нельзя использовать `len(self.data)`: это число столбцов, а не строк. Возьмите любой один столбец из словаря и найдите его длину.

> **Подсказка к `filter`.** Маска применяется к каждому столбцу отдельно. Можно построить новый словарь через dictionary comprehension и затем создать `ExoplanetDataset` из этого словаря.

> **Подсказка к `from_csv`.** Удобная последовательность: `csv.DictReader` → список строк-словарей → словарь списков по колонкам → `_parse_column` для каждой колонки → `return cls(data)`. Метод должен возвращать экземпляр класса, поэтому здесь и полезен `@classmethod`.

In [ ]:
class ExoplanetDataset:
    def __init__(self, data):
        # TODO: проверка dict, преобразование к np.ndarray,
        # одинаковая длина столбцов, копии массивов.
        pass

    def __len__(self):
        pass

    def __repr__(self):
        pass

    @property
    def columns(self):
        pass

    @property
    def shape(self):
        pass

    def __getitem__(self, column):
        pass

    def get_row(self, index):
        pass

    def filter(self, mask):
        # TODO: вернуть НОВЫЙ ExoplanetDataset.
        pass

    def add_column(self, name, values):
        # TODO: вернуть НОВЫЙ ExoplanetDataset.
        pass

    def _numeric(self, column):
        pass

    def mean(self, column):
        pass

    def median(self, column):
        pass

    def std(self, column):
        pass

    def count_missing(self, column):
        pass

    def describe(self, column):
        # dict: count, missing, mean, std, min, median, max
        pass

    def unique(self, column):
        pass

    def value_counts(self, column):
        pass

    def standardize(self, column):
        pass

    @staticmethod
    def _parse_column(values):
        # Дополнительное задание Попробуйте преобразовать столбец в float.
        # Пустые числа -> np.nan.
        # Если столбец строковый: пустые значения -> None.
        pass

    @classmethod
    def from_csv(cls, filename):
        # Дополнительное задание csv.DictReader -> столбцы -> _parse_column -> cls(data)
        pass

### Тесты класса на искусственных данных

In [ ]:
ds = ExoplanetDataset(dirty_data)

assert len(ds) == 1000
assert ds.shape == (1000, 7)
assert "pl_rade" in ds.columns
assert ds["pl_rade"].shape == (1000,)
assert ds.get_row(0)["pl_name"] == "SIM-0000 b"

print(ds)
print("OK: Базовая структура класса работает")

In [ ]:
small_ds = ds.filter(ds["pl_rade"] < 2)

assert isinstance(small_ds, ExoplanetDataset)
assert len(small_ds) < len(ds)
assert len(ds) == 1000

ds2 = ds.add_column("dummy_feature", np.arange(len(ds)))
assert "dummy_feature" in ds2.columns
assert "dummy_feature" not in ds.columns

print("OK: filter и add_column работают")

In [ ]:
assert ds.count_missing("pl_rade") == 3

desc = ds.describe("pl_rade")
expected_keys = {"count", "missing", "mean", "std", "min", "median", "max"}
assert set(desc) == expected_keys
assert desc["missing"] == 3

standardized = ds.standardize("st_mass")
assert np.isclose(np.nanmean(standardized), 0.0)
assert np.isclose(np.nanstd(standardized), 1.0)

print("OK: Статистические методы работают")

### Дополнительное задание Задание 4. ООП — объясните словами

1. Что хранится в состоянии объекта `ExoplanetDataset`?
2. Почему `dataset["pl_rade"]` начинает работать после реализации `__getitem__`?
3. Почему `filter` возвращает новый объект?
4. Чем `@classmethod from_csv` удобен как альтернативный конструктор?

**Ответ:**

1. ...
2. ...
3. ...
4. ...

## 5. Настоящие данные: NASA Exoplanet Archive

Используем `Planetary Systems Composite Parameters (pscomppars)`.
Для учебной работы берём 15 колонок.

Документация:
- https://exoplanetarchive.ipac.caltech.edu/docs/TAP/usingTAP.html
- https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html

**Преподавателю:** включите скачивание один раз и затем раздайте
`nasa_exoplanets.csv` вместе с ноутбуком. Тогда у всех учеников будут одинаковые данные.

### Материалы к разделу

- [Python `csv` module](https://docs.python.org/3/library/csv.html) — официальная документация по `csv.reader` и `csv.DictReader`.
- [Real Python: Reading CSVs with Python's csv module, video](https://realpython.com/videos/reading-csvs-pythons-csv-module/) — короткий разбор `DictReader`.
- [NASA Exoplanet Archive: TAP documentation](https://exoplanetarchive.ipac.caltech.edu/docs/TAP/usingTAP.html) — как программно запрашиваются таблицы архива.
- [NASA Exoplanet Archive: PS/PSCompPars columns](https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html) — физический смысл полей каталога.

При чтении CSV помните: `csv.DictReader` возвращает значения как строки. Поэтому числовые колонки необходимо преобразовать отдельно.

> **Подсказка к парсингу CSV.** Не определяйте тип по одному элементу: первая строка в столбце может быть пустой. Попробуйте преобразовать все непустые значения в `float`; если хотя бы одно непустое значение не преобразуется, считайте столбец строковым.

In [ ]:
NASA_COLUMNS = [
    "pl_name", "hostname", "discoverymethod", "disc_year",
    "pl_orbper", "pl_orbsmax", "pl_rade", "pl_bmasse",
    "pl_eqt", "pl_insol", "st_teff", "st_rad", "st_mass",
    "sy_dist", "sy_pnum",
]

NASA_CSV_PATH = Path("nasa_exoplanets.csv")

query = "select " + ",".join(NASA_COLUMNS) + " from pscomppars"
params = urlencode({"query": query, "format": "csv"})
NASA_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?" + params

DOWNLOAD_NASA_DATA = False

if DOWNLOAD_NASA_DATA:
    urlretrieve(NASA_URL, NASA_CSV_PATH)
    print("Сохранено:", NASA_CSV_PATH.resolve())
else:
    print("Автоскачивание выключено.")
    print("Для создания снимка преподавателем: DOWNLOAD_NASA_DATA = True")

### Задание 5. Загрузите NASA-каталог

Реализуйте `from_csv`, положите `nasa_exoplanets.csv` рядом с ноутбуком и загрузите его.

In [ ]:
if NASA_CSV_PATH.exists():
    nasa = ExoplanetDataset.from_csv(NASA_CSV_PATH)
    print(nasa)
    print(nasa.columns)
    print(nasa.get_row(0))
else:
    print("Файл nasa_exoplanets.csv не найден.")

## 6. EDA без pandas и Polars

### Задание 6

1. Сколько планет в снимке?
2. Для каких пяти числовых колонок доля пропусков максимальна?
3. Найдите медианные радиус и период.
4. Найдите пять планет с наименьшим известным радиусом.
5. Найдите пять планет с самым коротким известным периодом.

### Материалы к разделу

- [NumPy boolean indexing](https://numpy.org/doc/stable/user/basics.indexing.html#boolean-array-indexing) — фильтрация строк по маске.
- [numpy.argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) — поиск объектов с минимальными или максимальными значениями.
- [NumPy statistics](https://numpy.org/doc/stable/reference/routines.statistics.html) — функции для описательной статистики.
- [NASA: Exoplanet Detection Methods](https://www.nasa.gov/universe/exoplanets/exoplanet-detection-methods/) — астрономический контекст методов обнаружения.

Для поиска пяти минимальных значений обычно удобно сначала убрать `NaN`, затем применить `np.argsort(...)[:5]`.

> **Подсказка к заданию 6.** Для числового столбца сначала создайте `valid = ~np.isnan(column)`. После этого сортируйте `column[valid]`; тем же индексом можно индексировать `pl_name[valid]`.

> **Подсказка к заданию 7.** После фильтра по году у вас снова объект `ExoplanetDataset`, поэтому к нему можно применить уже написанный `value_counts`.

In [ ]:
if NASA_CSV_PATH.exists():
    # TODO
    pass

### Задание 7. Методы открытия

Получите уникальные методы открытия, частоты и отсортируйте их по убыванию.
Дополнительное задание Определите самый частый метод среди планет с `disc_year >= 2015`.

In [ ]:
if NASA_CSV_PATH.exists():
    # TODO
    pass

### Задание 8. Учебный поиск «похожих на Землю по двум параметрам»

Отберите:
- `0.8 <= pl_rade <= 1.5`;
- `200 <= pl_eqt <= 350`.

Выведите название, радиус и температуру.

> Это не критерий обитаемости, а только упражнение на фильтрацию.

In [ ]:
if NASA_CSV_PATH.exists():
    # TODO
    pass

## 7. Физика + NumPy

### Задание 9. Относительная средняя плотность Дополнительное задание

$
\frac{\rho_p}{\rho_\oplus} = \frac{M_p/M_\oplus}{(R_p/R_\oplus)^3}.
$

Вычислите `relative_density`, добавьте её в новый датасет и найдите пять максимальных
известных значений. Объясните, почему экстремумы надо интерпретировать осторожно.

### Материалы к разделу

- [NASA JPL: Exploring Exoplanets with Kepler](https://www.jpl.nasa.gov/edu/resources/lesson-plan/exploring-exoplanets-with-kepler/) — школьное объяснение третьего закона Кеплера и его применения к экзопланетам.
- [NASA Science: Orbits and Kepler's Laws](https://science.nasa.gov/solar-system/orbits-and-keplers-laws/) — физический смысл трёх законов Кеплера.
- [NASA Exoplanet Archive: column definitions](https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html) — единицы `pl_orbper`, `pl_orbsmax`, `pl_bmasse`, `pl_rade`, `st_mass`.
- [NumPy indexing](https://numpy.org/doc/stable/user/basics.indexing.html) — создание маски только для строк, где нужные величины известны.

Перед подстановкой в формулу обязательно проверьте единицы: период в каталоге задан в сутках, а в используемой форме закона Кеплера нужен период в годах.

> **Подсказка к плотности.** Сначала создайте маску строк, где одновременно известны масса и радиус. Затем заведите массив нужной длины, заполненный `np.nan`, и присвойте вычисленные значения только позициям `valid`.

> **Подсказка к закону Кеплера.** Не забудьте `P_obs = pl_orbper / 365.25`. Массив `valid` должен одновременно проверять отсутствие `NaN` и положительность периода, большой полуоси и массы звезды.

In [ ]:
if NASA_CSV_PATH.exists():
    # TODO
    pass

### Задание 10. Проверяем третий закон Кеплера Задание повышенной сложности

$
P_{\mathrm{year}}^2 \approx \frac{a^3}{M_\star}.
$

Для строк с известными `pl_orbper`, `pl_orbsmax`, `st_mass`:
переведите период в годы, вычислите предсказанный период и относительную ошибку

$
error = \frac{|P_{\mathrm{pred}} - P_{\mathrm{obs}}|}{P_{\mathrm{obs}}}.
$

Найдите медианную ошибку и пять объектов с максимальной ошибкой.
Затем предложите минимум две причины расхождений.

In [ ]:
if NASA_CSV_PATH.exists():
    # TODO
    pass

**Причины расхождений:**

1. ...
2. ...

## 8. Финальное мини-исследование

В 5–8 предложениях ответьте:

1. Какие признаки заполнены хуже всего?
2. Какой метод открытия доминирует?
3. Чем отличается полнота данных по радиусам и массам?
4. Какие объекты прошли учебный фильтр по радиусу и температуре?
5. Насколько хорошо выполняется проверка третьего закона Кеплера?
6. Назовите два ограничения анализа.

### Материалы к разделу

- [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/) — исходный научный каталог, с которым вы работаете.
- [NASA Exoplanet Resource Guide](https://www.nasa.gov/stem-content/exoplanet-resource-guide/) — материалы о методах поиска и свойствах экзопланет.
- [NumPy statistics](https://numpy.org/doc/stable/reference/routines.statistics.html) — если нужно перепроверить используемые статистики.

Финальный текст должен опираться на полученные числа. Разделяйте наблюдение в данных и физическую интерпретацию: например, большая доля транзитных открытий не означает, что во Вселенной большинство планет обязательно обнаружимо именно этим методом.

> **Подсказка к выводу.** Хорошая структура: сначала 2–3 конкретных результата с числами, затем их осторожная интерпретация, затем ограничения данных и метода. Не превращайте учебный фильтр по радиусу и температуре в утверждение об обитаемости.

**Вывод:**

...

## 9. Что нужно сдать

Заполненный `.ipynb`, где проходят тесты, выполнен анализ NASA и написан итоговый вывод.

В следующем ноутбуке мы сравним собственный класс с **Polars** и добавим
визуализацию через **Matplotlib**.

### Материалы к разделу

- [Jupyter Notebook: running code](https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Running%20Code.html) — запуск ячеек и перезапуск kernel.
- [Jupyter Notebook basic workflow](https://github.com/jupyter/notebook/blob/main/docs/source/notebook.md#basic-workflow) — как устроена работа с состоянием ноутбука.

Перед сдачей рекомендуется перезапустить kernel и выполнить все ячейки сверху вниз. Это позволяет обнаружить ситуации, когда код случайно зависит от переменной, созданной в другой последовательности запуска.